# 🌊 04 · The Frequency Domain and Scale
### Seeing an image as a sum of spatial frequencies, and why scale matters

**Part B · Seeing like a microscope** · Module 04 of 29 · *BiologyToBioimage*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/sci_img/tutorial/BiologyToBioimage/notebooks/04_frequency_domain_and_scale.ipynb)

| | |
|---|---|
| **Hardware target** | CPU only (no GPU needed) |
| **Core frameworks** | NumPy (FFT), Matplotlib |
| **Prerequisites** | Modules 02–03 |
| **Estimated time** | 60–75 minutes |
| **02-740 topics covered** | The 2-D Fourier transform; low-pass/high-pass filtering in the frequency domain; image pyramids and scale-space |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> Every spectrum and pyramid below is computed from a SIMULATED image generated in this notebook.

## 🎯 Learning objectives

1. Explain, intuitively, what it means to re-express an image in terms of spatial frequency rather
   than spatial position.
2. Compute and visualize the 2-D Fourier magnitude spectrum of a synthetic image.
3. Implement low-pass and high-pass filtering directly in the frequency domain, and connect this to
   the spatial-domain smoothing of Module 03.
4. Build a simple image pyramid and explain why "scale" is a dimension algorithms must often search
   over, not just space.

## 🗺️ Contents

1. What is a spatial frequency?
2. The 2-D Fourier transform and the magnitude spectrum
3. Low-pass and high-pass filtering in the frequency domain
4. Image pyramids and scale-space

## 🧫 The biological question

A periodic scan-line artifact in a camera, the fine texture of chromatin inside a nucleus, and the
smooth, slowly varying shading of an uneven illumination field are all, in a real sense, different
*spatial frequencies* superimposed in the same image. The spatial domain (the array of pixel
values you have worked with so far) makes it easy to ask "how bright is this pixel?" but hard to ask
"how much of this image's content is fine, repetitive texture versus slow, smooth shading?" The
frequency domain answers exactly that second kind of question, and it will turn out to be essential
groundwork for denoising and super-resolution (Module 16), restoration (Module 17), and even
intensity-based registration (Module 15).

---
## Environment setup

In [ ]:
# ── Hardware check ──
import platform
print('Python', platform.python_version())
try:
    import torch
    print('Torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
except ImportError:
    print('PyTorch not installed yet — the next cell installs it if needed.')


In [ ]:
!pip install -q scikit-image
# ── Dependencies ── (Colab already has most of these; we install only what is missing)


In [ ]:
# ── Global imports and configuration ──
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
SEED = 4
rng = np.random.default_rng(SEED)
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.titlesize'] = 11


### 🧰 Course toolkit

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.0; copied so this notebook runs alone) ──
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from scipy.optimize import linear_sum_assignment

def gaussian_psf(sigma: float, radius: int | None = None) -> np.ndarray:
    """A normalised 2-D Gaussian point-spread function (the microscope's blur kernel)."""
    if radius is None:
        radius = max(1, int(np.ceil(3 * sigma)))
    ax = np.arange(-radius, radius + 1)
    xx, yy = np.meshgrid(ax, ax)
    k = np.exp(-(xx ** 2 + yy ** 2) / (2 * sigma ** 2))
    return k / k.sum()


def blur_image(image: np.ndarray, psf: np.ndarray) -> np.ndarray:
    """Convolve an image with a point-spread function (microscope optical blur)."""
    return ndi.convolve(image, psf, mode="reflect")


def add_gaussian_noise(image: np.ndarray, sigma: float, rng: np.random.Generator) -> np.ndarray:
    """Additive Gaussian (read) noise: the classic noise floor of a camera sensor."""
    return image + rng.normal(0.0, sigma, size=image.shape)


def add_periodic_noise(image: np.ndarray, amplitude: float, freq: tuple[float, float]) -> np.ndarray:
    """Periodic (structured) noise: a sinusoidal stripe pattern, as from scan-line pickup."""
    h, w = image.shape
    yy, xx = np.meshgrid(np.arange(h), np.arange(w), indexing="ij")
    return image + amplitude * np.sin(2 * np.pi * (freq[0] * yy / h + freq[1] * xx / w))


SHAPE_CLASSES = ("round", "elongated", "kidney", "ring", "dividing", "irregular")


def _draw_blob(canvas: np.ndarray, cy: float, cx: float, r: float, shape: str,
                rng: np.random.Generator, angle: float = 0.0) -> None:
    """Add one synthetic nucleus/cell of the given shape class onto ``canvas`` (in place, additive)."""
    size = canvas.shape[0]
    yy, xx = np.meshgrid(np.arange(size), np.arange(size), indexing="ij")
    y, x = yy - cy, xx - cx
    ca, sa = np.cos(angle), np.sin(angle)
    yr, xr = ca * y + sa * x, -sa * y + ca * x  # rotate into the blob's own frame
    if shape == "round":
        d = (yr / r) ** 2 + (xr / r) ** 2
        canvas += np.exp(-d * 2.0)
    elif shape == "elongated":
        d = (yr / (r * 0.55)) ** 2 + (xr / (r * 1.7)) ** 2
        canvas += np.exp(-d * 2.0)
    elif shape == "kidney":
        d = (yr / r) ** 2 + (xr / r) ** 2
        dent = np.exp(-((yr - 0.0) ** 2 + (xr - r * 0.9) ** 2) / (2 * (r * 0.5) ** 2))
        canvas += np.clip(np.exp(-d * 2.0) - 0.9 * dent, 0, None)
    elif shape == "ring":
        d = np.sqrt(yr ** 2 + xr ** 2)
        canvas += np.exp(-((d - r) ** 2) / (2 * (r * 0.28) ** 2))
    elif shape == "dividing":
        sep = r * 1.1
        for sign in (-1, 1):
            d = ((yr - sign * sep * 0.5) / (r * 0.75)) ** 2 + (xr / (r * 0.75)) ** 2
            canvas += np.exp(-d * 2.0)
    else:  # irregular: a round blob perturbed by a few random bumps
        d = (yr / r) ** 2 + (xr / r) ** 2
        base = np.exp(-d * 2.0)
        for _ in range(3):
            bx, by = rng.uniform(-r, r, size=2)
            bump = np.exp(-(((yr - by) ** 2 + (xr - bx) ** 2) / (2 * (r * 0.45) ** 2)))
            base = np.clip(base + 0.5 * bump, 0, 1.3)
        canvas += base


def simulate_nuclei_image(n_cells: int = 25, size: int = 160, touching: bool = False,
                           shape_classes: tuple[str, ...] | None = None,
                           psf_sigma: float = 1.2, background: float = 0.05,
                           noise_sigma: float = 0.03, seed: int = 0):
    """A SIMULATED single-channel fluorescence image of cell nuclei/cells.

    Returns ``image`` (size x size, background + PSF-blurred blobs + Gaussian noise),
    ``instance_mask`` (0 = background, 1..n_cells = one integer id per cell, the ground truth
    for segmentation/detection/tracking), ``centroids`` (n_cells x 2, row, col), ``shape_labels``
    (one of SHAPE_CLASSES per cell, the ground truth for classification), and ``boxes`` (n_cells x 4,
    ymin, xmin, ymax, xmax — the ground truth for detection).
    """
    rng = np.random.default_rng(seed)
    classes = shape_classes or SHAPE_CLASSES
    image = np.full((size, size), background, dtype=float)
    instance_mask = np.zeros((size, size), dtype=int)
    centroids, labels, boxes = [], [], []
    margin = size * 0.12
    min_sep = size * 0.11 if not touching else size * 0.065
    placed = []
    attempts = 0
    while len(placed) < n_cells and attempts < n_cells * 60:
        attempts += 1
        cy, cx = rng.uniform(margin, size - margin, size=2)
        if all(np.hypot(cy - py, cx - px) > min_sep for py, px in placed):
            placed.append((cy, cx))
    for i, (cy, cx) in enumerate(placed):
        shape = classes[rng.integers(len(classes))]
        r = rng.uniform(size * 0.028, size * 0.045)
        angle = rng.uniform(0, 2 * np.pi)
        single = np.zeros_like(image)
        _draw_blob(single, cy, cx, r, shape, rng, angle)
        image += single
        this_mask = single > 0.25
        instance_mask[this_mask & (instance_mask == 0)] = i + 1
        centroids.append((cy, cx))
        labels.append(shape)
        ys, xs = np.where(this_mask)
        if len(ys):
            boxes.append((ys.min(), xs.min(), ys.max(), xs.max()))
        else:
            boxes.append((cy - r, cx - r, cy + r, cx + r))
    image = blur_image(image, gaussian_psf(psf_sigma))
    image = add_gaussian_noise(image, noise_sigma, rng)
    image = np.clip(image, 0, None)
    return dict(image=image, instance_mask=instance_mask, centroids=np.array(centroids),
                shape_labels=np.array(labels), boxes=np.array(boxes), source="SIMULATED")


def fft_lowpass_denoise(image: np.ndarray, cutoff_frac: float = 0.15) -> np.ndarray:
    """Denoise by zeroing high spatial frequencies in the Fourier domain (an ideal low-pass
    filter at ``cutoff_frac`` of the Nyquist frequency), then inverse-transforming."""
    f = np.fft.fftshift(np.fft.fft2(image))
    h, w = image.shape
    yy, xx = np.meshgrid(np.arange(h) - h / 2, np.arange(w) - w / 2, indexing="ij")
    radius = np.hypot(yy, xx) / (min(h, w) / 2)
    f[radius > cutoff_frac] = 0
    return np.real(np.fft.ifft2(np.fft.ifftshift(f)))


def show_image(ax, image: np.ndarray, title: str = "", cmap: str = "gray"):
    """Show one image on a given Matplotlib axis, titled, with ticks removed."""
    ax.imshow(image, cmap=cmap)
    ax.set_title(title, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])


def show_image_grid(images, titles=None, ncols: int = 4, cmap: str = "gray", figsize_scale: float = 2.6):
    """A grid of images with Matplotlib, one subplot each, titles optional."""
    n = len(images)
    ncols = min(ncols, n)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(figsize_scale * ncols, figsize_scale * nrows))
    axes = np.atleast_1d(axes).ravel()
    for i, img in enumerate(images):
        show_image(axes[i], img, titles[i] if titles else "", cmap=cmap)
    for ax in axes[n:]:
        ax.axis("off")
    fig.tight_layout()
    return fig, axes

---
## 1 · What is a spatial frequency?

A 1-D signal that oscillates slowly (few cycles per unit length) has *low* spatial frequency; one
that oscillates rapidly (many cycles per unit length) has *high* spatial frequency. Any signal — and,
by direct extension, any 2-D image — can be written as a weighted sum of pure sinusoids of different
frequencies, orientations, and phases. This is the central claim of **Fourier analysis**: instead of
describing an image pixel-by-pixel (the spatial domain), you can describe it frequency-by-frequency
(the frequency domain) — the same information, re-expressed in a different, often more useful,
basis.

In [ ]:
# ── 1.1 build a 2-D image as an explicit sum of a few sinusoids, to build intuition ──
size = 128
yy, xx = np.meshgrid(np.arange(size), np.arange(size), indexing='ij')

low_freq = np.sin(2 * np.pi * 3 * xx / size)          # 3 cycles across the image: LOW frequency
high_freq = np.sin(2 * np.pi * 25 * xx / size)        # 25 cycles across the image: HIGH frequency
diagonal_freq = np.sin(2 * np.pi * (8 * xx + 8 * yy) / size)  # a diagonal, mid frequency

combined = 0.5 * low_freq + 0.3 * high_freq + 0.4 * diagonal_freq

fig, axes = plt.subplots(1, 4, figsize=(11, 3))
for ax, (title, arr) in zip(axes, [('low frequency\n(3 cycles)', low_freq), ('high frequency\n(25 cycles)', high_freq),
                                     ('diagonal, mid freq.', diagonal_freq), ('sum of all three', combined)]):
    show_image(ax, arr, title)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Three pure sinusoidal patterns — differing only in how many cycles
they complete across the image, and in orientation — summed together to produce a more complex,
textured-looking pattern. Fourier analysis runs this process in reverse: given the complicated
`combined` image, it recovers exactly which sinusoidal frequencies (and how much of each) were
present, by construction.

---
## 2 · The 2-D Fourier transform and the magnitude spectrum

The 2-D discrete Fourier transform converts an $H \times W$ image into an equally sized array of
complex numbers, one per spatial-frequency pair $(u, v)$. The **magnitude spectrum**
$|F(u,v)|$ tells you how much of each frequency is present (the **phase** carries positional
information we will not dwell on here). By convention, after `np.fft.fftshift`, the center of the
displayed spectrum corresponds to zero frequency (slowly varying / constant content), and distance
from the center corresponds to increasingly high spatial frequency.

In [ ]:
# ── 2.1 compute and display the magnitude spectrum of our combined sinusoids, and of a real nuclei image ──
def magnitude_spectrum(image):
    f = np.fft.fftshift(np.fft.fft2(image))
    return np.log1p(np.abs(f))   # log scale: spectra span many orders of magnitude

sim = simulate_nuclei_image(n_cells=15, size=size, seed=SEED)

fig, axes = plt.subplots(2, 2, figsize=(8, 8))
show_image(axes[0, 0], combined, 'synthetic sinusoid mixture')
show_image(axes[0, 1], magnitude_spectrum(combined), 'its magnitude spectrum', cmap='inferno')
show_image(axes[1, 0], sim['image'], 'simulated nuclei image')
show_image(axes[1, 1], magnitude_spectrum(sim['image']), 'its magnitude spectrum', cmap='inferno')
plt.tight_layout(); plt.show()

🧪 **Ground-truth check.** In the top-right spectrum, look for three pairs of bright dots
symmetric about the center — these correspond exactly to the three sinusoids we built by hand (the
Fourier transform of a real image is always conjugate-symmetric, which is why frequencies appear in
mirrored pairs). You placed those frequencies yourself in Section 1, so you can directly verify the
spectrum is finding exactly what you put there. The bottom-right spectrum (a real, non-sinusoidal
nuclei image) instead shows a bright central blob (the image's slowly-varying overall brightness and
large-scale structure) fading outward (its finer, noisier detail) — a typical "natural image"
spectrum shape.

✏️ **Your turn.** Add a fourth, even-higher-frequency sinusoid (say, 50 cycles across the image)
to `combined` with a small weight (e.g. 0.1), recompute its magnitude spectrum, and find the new
bright dot pair it produces. How far from the center is it, compared to the 25-cycle component's
dots?

---
## 3 · Low-pass and high-pass filtering in the frequency domain

Filtering in the frequency domain is remarkably simple conceptually: to **low-pass filter**
(smooth, suppress fine detail and noise), zero out the Fourier coefficients far from the center
(high spatial frequencies) and inverse-transform back. To **high-pass filter** (enhance edges and
fine detail, suppress slow shading), do the opposite: zero out the coefficients *near* the center
(low spatial frequencies). This is the exact frequency-domain counterpart of the spatial-domain
Gaussian smoothing from Module 03 — in fact, convolution in the spatial domain is *mathematically
equivalent* to multiplication in the frequency domain, a fact known as the **convolution theorem**:
$g = \omega * f \iff G(u,v) = W(u,v)\cdot F(u,v)$.

In [ ]:
# ── 3.1 periodic noise is a textbook case where frequency-domain filtering shines ──
clean = sim['image']
periodic_noisy = add_periodic_noise(clean, amplitude=0.25, freq=(0.0, 6.0))  # a scan-line-like stripe artifact

denoised = fft_lowpass_denoise(periodic_noisy, cutoff_frac=0.12)

fig, axes = plt.subplots(2, 3, figsize=(11, 7))
show_image(axes[0, 0], clean, 'clean')
show_image(axes[0, 1], periodic_noisy, 'with periodic (stripe) noise')
show_image(axes[0, 2], denoised, 'after FFT low-pass')
show_image(axes[1, 0], magnitude_spectrum(clean), 'spectrum: clean', cmap='inferno')
show_image(axes[1, 1], magnitude_spectrum(periodic_noisy), 'spectrum: noisy (note bright dots!)', cmap='inferno')
show_image(axes[1, 2], magnitude_spectrum(denoised), 'spectrum: denoised', cmap='inferno')
plt.tight_layout(); plt.show()

🔍 **What just happened?** The periodic stripe artifact shows up in the frequency domain as two
sharp, isolated bright dots at a specific frequency (visible in the middle spectrum) — exactly
because a pure sinusoid concentrates almost all of its energy at one frequency pair. A simple
low-pass filter (zeroing everything beyond a cutoff radius) happens to remove it here along with
other high-frequency content, which is why this toy approach is somewhat blunt; a more surgical
real-world fix (called a *notch filter*) would zero out only those two specific bright dots and
leave the rest of the spectrum, and therefore the rest of the fine real detail, untouched.

✏️ **Your turn.** Implement a tiny high-pass version of `fft_lowpass_denoise` (zero out
everything *inside* a cutoff radius instead of outside it) and apply it to `clean`. What kind of
image content survives? Does it look like an edge-detection result? (This previews Module 06.)

### 📘 Lecture notes — the frequency domain

**Definition.** The 2-D discrete Fourier transform re-expresses an image as a weighted sum of 2-D
sinusoids at different spatial frequencies and orientations; the convolution theorem states that
spatial-domain convolution equals frequency-domain multiplication, linking every filtering operation
from Module 03 to an equivalent frequency-domain operation.

**Worked example.** A fast Fourier transform (FFT) algorithm computes this transform in
$O(N \log N)$ time rather than the $O(N^2)$ a naive implementation would require, which is precisely
why frequency-domain filtering of large microscopy images is computationally practical.

**One memorable point.** *A sharp, isolated, periodic artifact (scan-line noise, a grid pattern)
usually hides in plain sight in the spatial domain but stands out as an unmistakable bright spot in
the frequency domain — when you suspect structured noise, look at the spectrum.*

---
## 4 · Image pyramids and scale-space

Many biological structures of interest come in a range of sizes — a single small vesicle, a whole
nucleus, an entire cell — and a single, fixed analysis scale can never be optimal for all of them
simultaneously (recall Module 03's low-pass-filter-sigma trade-off: too small a sigma misses large
structures, too large a sigma destroys small ones). An **image pyramid** builds a sequence of
progressively smoothed-and-downsampled versions of the same image, giving you — in one data
structure — access to the image at many different scales at once. This multiscale idea, called
**scale-space**, underlies the choice of sigma for point/particle detection in Module 05 and the
scale-selection discussion for edge detection in Module 06.

In [ ]:
# ── 4.1 build a simple Gaussian image pyramid by repeated smoothing + downsampling ──
def gaussian_pyramid(image, n_levels=4, sigma=1.0):
    levels = [image]
    current = image
    for _ in range(n_levels - 1):
        smoothed = blur_image(current, gaussian_psf(sigma))
        current = smoothed[::2, ::2]   # downsample by 2x in each axis
        levels.append(current)
    return levels

pyramid = gaussian_pyramid(sim['image'], n_levels=4, sigma=1.0)
fig, axes = plt.subplots(1, len(pyramid), figsize=(11, 3))
for ax, (lvl, img) in zip(axes, enumerate(pyramid)):
    show_image(ax, img, f'level {lvl}\n{img.shape[0]}x{img.shape[1]}')
plt.tight_layout(); plt.show()

🔍 **What just happened?** Each pyramid level is a coarser, smaller version of the same image —
individual small nuclei become indistinguishable blobs by level 2–3, while the overall arrangement
of cells across the field of view remains visible even at the coarsest level. An algorithm that
needs to find both small and large structures can search each pyramid level with the *same*,
scale-appropriate detector rather than needing one detector tuned simultaneously to every possible
size.

## 📌 Key takeaways

- Any image can be re-expressed as a sum of 2-D sinusoids at different spatial frequencies — the frequency domain is a different, equally valid representation of the same information.
- The convolution theorem links spatial-domain filtering (Module 03) and frequency-domain filtering: convolution in one domain is multiplication in the other.
- Periodic, structured noise is often easiest to spot and remove by looking at the magnitude spectrum, where it appears as isolated bright peaks.
- Image pyramids provide access to multiple spatial scales at once, which matters whenever biological structures of interest vary substantially in size.

## ✅ Checkpoint

**1.** What does a bright, isolated pair of dots far from the center of a magnitude spectrum usually
indicate about the original image?

<details><summary>Answer</summary>
A strong, nearly pure sinusoidal (periodic) component at a specific spatial frequency and
orientation — often a structured artifact like scan-line or grid noise, rather than genuine
biological texture.
</details>

**2.** State the convolution theorem in one sentence.

<details><summary>Answer</summary>
Convolving two signals in the spatial domain is equivalent to multiplying their Fourier transforms
in the frequency domain (and vice versa).
</details>

**3.** Why might you want to analyze an image at multiple scales (a pyramid) instead of a single
fixed scale?

<details><summary>Answer</summary>
Because biological structures of interest often span a wide range of sizes, and no single fixed
smoothing/detection scale is simultaneously appropriate for both the smallest and largest structures
of interest.
</details>

## 🧠 Questions & discussion

- The low-pass filter we used to remove periodic noise also blurred away some genuine fine detail.
  Can you think of a way to design a filter that removes *only* the specific periodic frequency
  without touching anything else? (This is the idea behind a notch filter.)
- Why does a Gaussian pyramid smooth the image *before* downsampling, rather than just downsampling
  directly? (Hint: think back to Module 02's discussion of aliasing.)
- If convolution and frequency-domain multiplication are mathematically equivalent, why would you
  ever choose to implement a filter in the frequency domain rather than directly as a spatial
  convolution?

## 🚀 Bonus — going further

- Implement a true notch filter: zero out only a small neighborhood around the specific bright dot
  pair caused by the periodic noise, leaving the rest of the spectrum untouched, and compare the
  result to the blunt low-pass filter used above.
- Build a Laplacian pyramid (the pyramid of *differences* between consecutive Gaussian pyramid
  levels) and show that summing it back up reconstructs the original image.
- Read about the Scale Invariant Feature Transform (SIFT) and note how it searches explicitly across
  a scale-space pyramid to find features that are stable regardless of an object's size in the
  image.

## 📝 Lab note

```text
Module: 04 — The Frequency Domain and Scale
Date:
A periodic-noise signature I would now recognize in a spectrum:
Why low-pass filtering has a cost, in my own words:
```

## 📚 References

1. Bracewell, R.N. (2000). *The Fourier Transform and Its Applications* (3rd ed.). McGraw-Hill.
2. Cooley, J.W., Tukey, J.W. (1965). An algorithm for the machine calculation of complex Fourier
   series. *Mathematics of Computation*, 19(90), 297–301.
3. Burt, P.J., Adelson, E.H. (1983). The Laplacian pyramid as a compact image code. *IEEE
   Transactions on Communications*, 31(4), 532–540.
4. Lindeberg, T. (1994). *Scale-Space Theory in Computer Vision*. Kluwer Academic Publishers.
5. Gonzalez, R.C., Woods, R.E. (2018). *Digital Image Processing* (4th ed.). Pearson, Ch. 4.

## ➡️ Next

Module 05 puts filtering and scale selection to direct biological use: detecting point/particle
features such as axonal transport vesicles, including sub-pixel localization — a complete, real
analysis pipeline built from the tools introduced so far.